# SNCP-PPO Social Navigation — Colab Notebook

End-to-end notebook for training and evaluating an **LTC + PPO** crowd-aware navigation policy on Google Colab.

## What you'll do

1. **Setup** — clone repo, install deps, mount Drive (optional)
2. **Smoke test** — verify the environment + model + training loop
3. **Train** — vectorized (parallel-env) curriculum training with multi-scenario holdout
4. **Evaluate** — characterize the trained policy on randomized scenarios
5. **Visualize** — trajectory plots + GIFs across all scenarios
6. **Analyze** — read training CSV, plot learning curves

## Architecture recap

- **Policy**: SNCPPolicy (3 LTC cells: temporal/spatial/node + attention + actor-critic heads)
- **Observation** (robot-local): robot_node (7), spatial_edges (**H×6** = position + relative velocity + **goal-direction unit vector** per pedestrian), temporal_edges (2)
- **Algorithm**: PPO with clipped value loss, GAE with truncation bootstrap, BPTT over LTC subsequences
- **Environment**: randomized layout — robot + pedestrians spawn at random antipodal points on a circle every episode (true generalization, not a memorized scene)
- **Training**: **vectorized** - N parallel envs collect 2048 transitions/update; v16 curriculum reaches N=10 and replays earlier density phases to reduce forgetting
- **Best-checkpoint metric**: `min(success across holdout scenarios)` — rewards generalists

## Colab tips

- **Runtime → Change runtime type → A100 (Colab Pro+)** recommended (~3-4h for a 2M-step run). Free-tier T4 works but slower.
- **Mount Drive (Section 1.4)** so checkpoints/logs persist if the session disconnects mid-run.
- The vectorized path batches N envs through one policy forward pass, so the GPU is used far better than the old single-env path — but env stepping (Social-Force sim) is still CPU-side, so more vCPUs (Pro+) also helps.

## 1. Setup

### 1.1 Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1.2 Clone repository

Replace `heimdilon` with your GitHub user if you forked the repo.

In [ ]:
import os
REPO_URL = 'https://github.com/heimdilon/sncp-ppo-crowdnav.git'
REPO_DIR = '/content/sncp-ppo-crowdnav'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull --rebase

%cd {REPO_DIR}
!ls -la

### 1.3 Install dependencies

PyTorch comes pre-installed on Colab; we just add `ncps`, `gymnasium`, and confirm versions.

In [ ]:
!pip install -q -r requirements.txt

import torch
print(f'torch    {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'         device: {torch.cuda.get_device_name(0)}')

import gymnasium, ncps, numpy, matplotlib
print(f'gymnasium {gymnasium.__version__}')
print(f'ncps      {ncps.__version__}')
print(f'numpy     {numpy.__version__}')
print(f'matplotlib {matplotlib.__version__}')

### 1.4 (Optional) Mount Google Drive

If you want checkpoints/logs to persist across Colab sessions, mount Drive and we'll symlink `checkpoints/` and `logs/` into a Drive folder.

**Skip this cell if you're just running a quick experiment.**

In [ ]:
USE_DRIVE = False  # set True to persist runs across Colab sessions
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sncp-ppo-crowdnav-runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_PROJECT_DIR}/logs', exist_ok=True)
    # Replace local dirs with symlinks to Drive
    for sub in ('checkpoints', 'logs'):
        local = f'{REPO_DIR}/{sub}'
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            # Backup existing local dir then symlink
            import shutil
            for f in os.listdir(local):
                src = f'{local}/{f}'
                dst = f'{DRIVE_PROJECT_DIR}/{sub}/{f}'
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
            shutil.rmtree(local)
        os.symlink(f'{DRIVE_PROJECT_DIR}/{sub}', local)
    print(f'Drive-backed dirs: {DRIVE_PROJECT_DIR}/{{checkpoints,logs}}')
else:
    print('Drive mount skipped (USE_DRIVE=False). Files will be lost when Colab session ends.')

## 2. Smoke tests

Three fast self-tests:
1. Environment reset/step + observation shapes
2. Model forward pass
3. 50-episode mini-training (verifies the full pipeline + new Path A changes)

In [ ]:
!python test_env.py

In [ ]:
!python test_model.py

In [ ]:
# 50-episode LEGACY single-env smoke training - verifies curriculum, holdout,
# value clipping, LR schedule, return normalization, KL early-stop, holdout
# best-checkpoint warmup/threshold/tie-break (#14), and per-update diagnostics
# line (ent / kl / std / rms). This is NOT the v16 vectorized replay run; the
# full v16 run is cell 14. Replay is intentionally 0 here so this old smoke stays
# a quick baseline check for the single-env path.
#
# Uses subprocess.run with a list-of-args instead of `!python ... \` so the
# IPython shell never has a chance to mangle multi-line continuations (which
# previously caused: argument --holdout_scenarios: invalid choice: ' ').
import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--episodes', '50',
    '--num_humans', '5',
    '--seed', '42',
    '--eval_freq', '25',
    '--holdout_episodes', '3',
    '--holdout_scenarios', 'easy', 'hard',
    '--update_freq', '5',
    '--log_freq', '10',
    '--curriculum_replay_ratio', '0.0',
    '--save_path', 'checkpoints/sncp_ppo_smoke.pt',
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')

## 3. Full training (v18 — paper-faithful goal-reward restoration)

**v18 changes only the reward function, restoring it to the paper (Ao et al. 2026, Eq 18).** The env is otherwise identical to v16: non-reactive pedestrians at TurtleBot3 speed parity (≤0.26 m/s), comfort `-6*I_sp`, sparse AutoNCP architecture, density 1→10, 20% replay, `max_time=50s`.

**Why this and not another max-time bump:** v16's dominant failure is **timeout, not collision** — at N=1 the robot times out 60% of the time with `I_sp≈0.009` (i.e. nothing to avoid). That is a pure goal-reaching breakdown. Its cause is in the reward: v15 had (a) halved the approach coefficient 2→1 and (b) added an ad-hoc heading penalty `-weight*|angle_diff|` (not in the paper) whose per-step cost (up to 0.157) exceeded the entire max per-step progress reward (0.065 at 0.26 m/s). The policy therefore optimised *heading* over *arrival* and stalled into timeouts.

**The single conceptual change is `r_g → paper Eq 18`:** approach coefficient `1 → 2` and the heading penalty removed. `comfort_coeff` stays 6.0 and `max_time` stays 50s on purpose, so this run isolates the reward fix; raising the time cap would mask freezing rather than cure it.

### Key arguments (v18)

| Argument | Meaning | v18 value |
|---|---|---|
| `--num_envs` | Parallel envs | **16** |
| `--horizon` | Steps per env per PPO update | **128** |
| `--total_steps` | Env-step budget: drives curriculum + run length | **2_500_000** |
| `--num_humans` | Final curriculum density | **10** |
| `--curriculum_replay_ratio` | Update windows replaying earlier phases | **0.20** |
| `--comfort_coeff` | Social-pressure penalty coefficient | **6.0** (unchanged) |
| `--max_time` | Episode cap | **50.0** (unchanged) |
| `--holdout_scenarios` | Monitored during training (circle = N=10) | **easy hard circle** |
| `--lr` / `--target_kl` | Base lr / KL early-stop | **5e-5** / **0.01** |
| reward `r_g` | Goal shaping (in `crowd_env.py`) | **paper Eq 18: 2·Δd, no heading term** |

~4h on an A100. Acceptance is behavioral, not just scalar: success should rise (especially easy/medium), **timeout rate should fall**, `I_sp` stays low under non-reactive pedestrians, and trajectories should still route around the crowd at high density (the fix adds goal urgency; it does not tell the robot to ignore people).

In [ ]:
# Preflight before spending A100 time. This checks the current v17 notebook config,
# fail-fast guard, evaluation pipeline wiring, and committed v15 baseline.
import subprocess, sys
cmd = [
    sys.executable, 'verify_v16_run_ready.py',
    '--output', 'eval_v18/run_readiness.md',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

from IPython.display import Markdown, display
with open('eval_v18/run_readiness.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))


In [ ]:
# Customize before running - v18: paper-faithful goal-reward restoration.
# The task/env is IDENTICAL to v16: non-reactive pedestrians, speed parity
# <=0.26 m/s, comfort -6*I_sp, sparse AutoNCP architecture, density 1->10,
# 20% replay. The ONLY change vs v16 is the reward function in crowd_env.py,
# which v18 restores to the paper (Eq 18):
#   * approach shaping 1*delta_distance -> 2*delta_distance, and
#   * the ad-hoc -weight*|angle_diff| heading penalty is REMOVED (not in paper).
# Why: v16 failed by TIMEOUT, not collision (N=1 timeout 60% with I_sp~0.009 ->
# nothing to avoid, a pure goal-reaching breakdown). The halved approach coeff +
# heading penalty had starved the progress signal below the per-step orientation
# cost, so the policy optimised heading over arrival and stalled. comfort_coeff
# stays 6.0 and max_time stays 50.0 on purpose: this run isolates the reward fix
# (raising max_time would mask freezing instead of curing it).
# Real success = success rate climbs (esp. easy/medium), timeout rate falls,
# I_sp stays low, and trajectories still route around the crowd at high density.
#
# Curriculum (1->10 pedestrians, 10/25/50/75% phases) and holdout best-checkpoint
# selection are driven by TOTAL ENV STEPS (--total_steps), not episodes.
NUM_ENVS = 16         # parallel envs; 8 if GPU/CPU memory is tight
HORIZON = 128         # steps per env per update -> NUM_ENVS*HORIZON transitions/update
TOTAL_STEPS = 2_500_000   # env-step budget; drives curriculum + run length (~4h on A100)
SEED = 42
LR = 5e-5             # lowered from 1e-4 to damp holdout oscillation (v7/v8)
TARGET_KL = 0.01      # tighter than 0.015 default -> steadier convergence
REPLAY_RATIO = 0.20    # keep v16 anti-forgetting replay
COMFORT_COEFF = 6.0    # unchanged from v16 (v18 isolates the goal-reward fix)
MAX_TIME = 50.0        # unchanged from v16; v18's fix is the reward (paper Eq 18), not the clock
SAVE_PATH = 'checkpoints/sncp_ppo_v18.pt'

import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--num_envs', str(NUM_ENVS),
    '--horizon', str(HORIZON),
    '--total_steps', str(TOTAL_STEPS),
    '--eval_freq_updates', '20',
    '--num_humans', '10',
    '--seed', str(SEED),
    '--lr', str(LR),
    '--lr_end_factor', '0.1',
    '--target_kl', str(TARGET_KL),
    '--curriculum_replay_ratio', str(REPLAY_RATIO),
    '--comfort_coeff', str(COMFORT_COEFF),
    '--max_time', str(MAX_TIME),
    '--holdout_scenarios', 'easy', 'hard', 'circle',
    '--holdout_episodes', '50',
    '--save_path', SAVE_PATH,
]
print('Running:', ' '.join(cmd))
print('=' * 80)
# Stream output line by line so we see progress live
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')
if p.returncode != 0:
    raise SystemExit(p.returncode)

### Resuming a previous Colab session

If Colab disconnected mid-training: with `USE_DRIVE=True`, your latest periodic checkpoint (`sncp_ppo_v3_colab_ep<N>.pt`) is still in Drive. There's no built-in resume CLI — easiest path is to just rerun from scratch with a different `--seed` (each seed gives a fresh trajectory). For exact resume support, see the **Roadmap** at the end of this notebook.

## 4. Evaluation

Run the one-command post-run pipeline for v17. It evaluates the hard-scenario density sweep at N=1/3/5/8/10, writes trajectory plots, compares against the committed v15 baseline, analyzes the newest training CSV for late collapse, and writes the final artifact verification report.

In [ ]:
CHECKPOINT = 'checkpoints/sncp_ppo_v18.pt'  # v18 paper-faithful reward-restoration checkpoint
EVAL_OUT = 'eval_v18'
EVAL_SEED = 100  # different from training seeds for fair eval
EVAL_EPISODES = 50  # match the committed eval_v15 baseline for direct comparison

# One-command post-run gate. This runs the hard-scenario N=1/3/5/8/10 density
# sweep, trajectory plots, v15 comparison, training-collapse diagnostics using
# the newest logs/training_*.csv, and final artifact verification. It exits
# nonzero only if the artifact verifier returns a hard fail.
import subprocess, sys
cmd = [
    sys.executable, 'run_post_eval.py',
    '--version', '18',
    '--densities', '1', '3', '5', '8', '10',
    '--scenario', 'hard',
    '--n_episodes', str(EVAL_EPISODES),
    '--seed', str(EVAL_SEED),
    '--trajectory_densities', '5', '10',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

from IPython.display import Image, Markdown, display
for name in ['artifact_verification.md', 'comparison_vs_v15.md', 'training_diagnostics.md', 'report.md']:
    with open(f'{EVAL_OUT}/{name}', 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
display(Image(f'{EVAL_OUT}/density_sweep.png'))


### Compare with shipped v2 baseline (if checkpoints included in repo)

In [ ]:
import os

if os.path.exists('checkpoints/sncp_ppo_v2.pt'):
    print('Evaluating v2 baseline (pre-Path-A, has known catastrophic-forgetting issues)')
    for scenario, n in [('easy', 1), ('hard', 5)]:
        print(f'\n--- v2 on {scenario}/{n}h ---')
        !python test_eval.py --checkpoint checkpoints/sncp_ppo_v2.pt \
            --num_humans {n} --scenario {scenario} --n_episodes 50 --seed 100 2>&1 | tail -8
else:
    print('v2 checkpoint not in repo. Train and save one, or clone the full repo with checkpoints.')

## 5. Visualize trajectories

Generate trajectory plots (PNG) and animated GIFs to inspect what the policy is doing visually.

In [ ]:
# Single trajectory plot — finds first successful episode out of 20 tries and plots it
!python visualize_trajectory.py \
    --checkpoint {CHECKPOINT} \
    --output trajectory_plot.png \
    --num_humans 5 \
    --scenario hard \
    --seed 42

from IPython.display import Image, display
display(Image('trajectory_plot.png'))

In [ ]:
# Animated GIF for a single scenario
!python visualize_trajectory_gif.py --checkpoint {CHECKPOINT}

from IPython.display import Image, display
import glob
gifs = sorted(glob.glob('*.gif'))
if gifs:
    print(f'Generated: {gifs}')
    display(Image(gifs[-1]))

In [ ]:
# All scenarios as separate GIFs (easy/medium/hard/extreme)
!python visualize_all_scenarios_gif.py --checkpoint {CHECKPOINT}

from IPython.display import Image, display
for sc in ['easy', 'medium', 'hard', 'extreme']:
    path = f'{sc}_trajectory.gif'
    if os.path.exists(path):
        print(f'\n--- {sc} ---')
        display(Image(path))

## 6. Training curves analysis

Plot the learning trajectory with per-scenario holdout lines and the generalist `min(success)` dashed line that drove best-checkpoint selection.

In [ ]:
import glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if not csv_files:
    print('No training CSVs found. Run section 3 first.')
else:
    latest_csv = csv_files[-1]
    print(f'Plotting: {latest_csv}')
    !python plot_training.py --csv {latest_csv} --output training_curves_colab.png --window 50
    from IPython.display import Image, Markdown, display
    display(Image('training_curves_colab.png'))
    for path in ['eval_v18/training_diagnostics.md', 'eval_v18/artifact_verification.md']:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                display(Markdown(f.read()))
        except FileNotFoundError:
            print(f'{path} not found. Run the v17 evaluation cell first.')


### Inspect CSV in pandas (optional)

In [ ]:
import pandas as pd, glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f'Rows: {len(df)}')
    print(f'Columns: {list(df.columns)}')
    print('\nPhase distribution:')
    print(df['scenario'].value_counts().sort_index())
    print('\nHoldout summary (per-scenario success rate at each eval):')
    holdout_cols = [c for c in df.columns if c.startswith('holdout_') and c.endswith('_success')]
    if holdout_cols:
        # Take rows where holdout changed (event points only)
        hdf = df[holdout_cols].drop_duplicates()
        hdf.index = df.loc[hdf.index, 'episode']
        print(hdf.tail(10))

## 7. Persist results

If you used Drive (Section 1.4), checkpoints + logs already there. Otherwise download key artifacts before the session ends.

In [ ]:
from google.colab import files
import glob, os, shutil

DOWNLOAD = False  # set True to trigger browser download dialogs
if DOWNLOAD:
    # Best checkpoint
    if os.path.exists(SAVE_PATH):
        files.download(SAVE_PATH)
    # Latest training CSV + plot
    for pattern in ['logs/training_*.csv', 'training_curves_colab.png']:
        for f in sorted(glob.glob(pattern))[-1:]:
            files.download(f)
    # Complete v17 evaluation bundle: readiness, sweep, trajectories, comparison,
    # training diagnostics, and artifact verification.
    if os.path.isdir('eval_v18'):
        archive = shutil.make_archive('eval_v18_artifacts', 'zip', 'eval_v18')
        files.download(archive)

## 8. Notes & roadmap (current: v18 paper-faithful reward restoration)

`README.md` is stale; use `AGENTS.md` plus the code as the current source of truth.

### Why v18 changed the reward (root-cause, not another tweak)

v6→v17 stayed at 30–56% success despite many single-variable tweaks (comfort 6→5, replay, max-time). The deeper cause is that the **reward had drifted away from the paper** while the **task had drifted to be harder than the paper's** (the paper uses ORCA-reactive pedestrians at 1.0 m/s; this repo uses non-reactive pedestrians at 0.26 m/s parity). The clearest, best-isolated symptom is the N=1 60% timeout (nothing to avoid → goal-reaching is itself broken).

v18 fixes the most strongly-evidenced cause: it restores the paper's goal reward `r_g` (Eq 18) — approach `1→2·Δd` and removes the non-paper heading penalty. Comfort, max-time, replay, and architecture are untouched so the result is interpretable.

### Roadmap after v18 (remaining paper divergences, ordered)

1. **Comfort term** — impl uses `-6*I_sp` with `I_sp` *unbounded* (per-human `1/d` capped at 10); the paper uses `-2*I_sp` with `I_sp ∈ [0,1]`. The unbounded spikes during exploration over-taught caution. v19 candidate: clamp `I_sp` to [0,1] (paper spec) and/or lower `comfort_coeff` toward 2.
2. **Crowd reactivity** — the paper's 93–95% at N=10–20 relies on ORCA pedestrians that reciprocally avoid the robot. Non-reactive pedestrians make high-density collisions partly unavoidable for a slow unilateral robot. v20 candidate: a reactive/ORCA-like crowd, the single biggest difficulty divergence.
3. **Action space** — `v∈[0,vpref]` (no reverse) is fine for a real TurtleBot but limits last-moment evasion; only revisit after 1–2 with action-trace evidence.
4. **Eval gate** — the `eval_report.py` "no-beeline" gate treats high nav-time as good. That is only true when the path is blocked; on a clear path a near-straight route is *correct*. Make the nav-time gate density-aware so it stops rewarding dawdling.

### Evaluation gates

After cell 15 produces `checkpoints/sncp_ppo_v18.pt`, run cell 18 and read `eval_v18/artifact_verification.md`, then:

1. Artifact verification should not be `fail`.
2. Success/collision/**timeout** across N=1/3/5/8/10 — expect timeout to drop at low/mid density.
3. Trajectories at N=5 and N=10 should still route around the crowd, not through it.
4. `I_sp` should remain low under non-reactive pedestrians.
5. Read nav-time *per density*: near-straight at N=1 is good (efficiency), detours at N=10 are good (avoidance).